In [ ]:
import sys,os,re
import numpy             as np
import matplotlib.pyplot as plt
import pandas            as pd
import seaborn           as sb
from source_code.galdist import galaxy_distribution
from source_code.gwdist import gw_distribution

from itertools import product
from copy      import deepcopy
from time      import time

from scipy.interpolate import interp1d
from scipy.integrate   import trapz

from getdist import plots,loadMCSamples,MCSamples
import warnings
warnings.filterwarnings('ignore')
from getdist.gaussian_mixtures import GaussianND
import matplotlib
from matplotlib import rc
from matplotlib.pyplot import cm
from matplotlib.colors import LogNorm
from samplers.samplers_interface import nautilus_interface,run_fisher

rc('text', usetex=True)
rc('font', family='serif')
matplotlib.rcParams.update({'font.size': 18})

red    = '#8e001c'
yellow = '#ffb302'

sidelegend = {'bbox_to_anchor': (1.04,0.5), 
              'loc': "center left",
              'frameon': False}


In [ ]:
fiducial = {'ombh2': 0.022445,
            'omch2': 0.1205579307,
            'ns': 0.96,
            'As': 2.12605e-09,
            'H0': 67.,
            'w': -1.,
            'wa': 0.,
            'tau': 0.05,
            'mnu': 0.06,
            'a0': - 0.007589,
            'a1' :  0.002008,
            'a2' : - 0.004127,
            'a3' :  0.002918,
            'a4' : -0.0006784,
            'omegab': 0.05,
            'sigma8': 0.84,
            'omegam' : 0.31,
            #'A_IA': 1.72,
            #'eta_IA': -0.41,
            'b0_poly': 0.830703,
            'b1_poly': 1.190547,
            'b2_poly': -0.928357,
            'b3_poly': 0.423292,
            'MG_flag': 1,
            'pure_MG_flag': 2,
            'musigma_par': 1,
            'DE_model': 0,
            'mu0': -1.23,
            'sigma0': -0.17,
           }
fiducial['logA'] = np.log(fiducial['As']*1.e+10)

In [ ]:

N_gw_configurations = [1e4, 5*1e4, 1e5, 5*1e5, 1e6]
sigma_dL_configurations = [0.1, 0.05, 0.01, 0.005]
obs = ['GWC', 'GWWL', 'GWs']

results = {ob: {} for ob in obs}
for ob in obs:
    for N_gw in N_gw_configurations:
        for sigma_dL in sigma_dL_configurations:
            case = f"$N_gw$={int(N_gw)}, $\sigma$={sigma_dL}, LSS $\times$ {ob}"
            output_path=f"chains_MG/muSigmaCDM_fisher_3x2pt_{ob}_fixednuis_Ngw{int(N_gw)}_sigma{sigma_dL}"
            
            results[ob][case] = {'GW_configuration': {'N_gw': N_gw, 'sigma_dL': sigma_dL},
                                 'root': output_path,
                                 'sampler': 'Fisher',
                                 'covmat':False,}





In [ ]:
plot_pars = ['ombh2','omch2','ns','logA','H0','mu0','sigma0']

In [ ]:
def analyze_results(name,info):
    analysis = deepcopy(info)
    
    print('')
    print('\x1b[1;31m Analyzing {} \x1b[0m'.format(name))
    
    if info['sampler'] == 'MH':
        sample = loadMCSamples(info['root'], settings={'ignore_rows': 0.3})
        sample.cool(info['temperature'])
    
        if info['Nchains']>1:
            print('R-1({}) with {:.0f}% of points ignored = {:.3f}'.format(name,30,
                                                                           sample.getGelmanRubin()))
        else:
            print('Single chain, no R-1 computed. Trust Cobaya and hope for the best')
    elif info['sampler'] == 'Fisher':
        fisher = pd.read_csv(info['root']+'_matrix.txt',header=0,sep='\s+')
        fisher.index = fisher.columns
        fisher_info = np.load(info['root']+'_info.npy',allow_pickle=True).item()
        #print(fisher)
        #plt.figure()
        #plt.title(name)
        #sb.heatmap(fisher)
        
        sample = GaussianND([fisher_info[par]['fiducial'] for par in fisher.columns], 
                            np.linalg.inv(fisher.values),
                            names=fisher.columns, 
                            labels=[fisher_info[par]['latex'] for par in fisher.columns]).MCSamples(10000)
        if info['covmat'] == True:
            covmat = np.linalg.inv(fisher.values)
            headers = list(fisher.columns)
            output_file_path = info['root']+'_covmat.covmat' 
            with open(output_file_path, 'w') as f:
    
                f.write('# ' + ' '.join(headers) + '\n')
                np.savetxt(f, covmat, fmt='%.6e', delimiter='\t')

    else:
        sys.exit('Unknown sampler: {}'.format(info['sampler']))
        
        
        
    analysis['MCsamples'] = sample
    analysis['bounds']    = sample.getTable(limit=1).tableTex()
    
    all_pars     = sample.getParamNames().list()
    labels       = sample.getParamNames().labels()
    primary_pars = sample.getParamNames().getRunningNames()


    analysis['means'] = {par:val for par,val in zip(all_pars,sample.getMeans())}
    #print("\nParameter Analysis:")
    analysis['relative_error_percent'] = {}
    for param in all_pars:
        mean = analysis['means'][param]
        
        density = sample.get1DDensity(param)  
        bounds = density.getLimits(1) 
         
        lower_error, upper_error, _, _ = bounds
        sigma = (upper_error - lower_error) / 2  
        relative_error_percent = (sigma / mean) 
        analysis['relative_error_percent'][param] = relative_error_percent
        
        #print(f"{param}: Mean = {mean:.3f}, 1-sigma Error = {sigma:.3f}, Relative Error = {relative_error_percent:.2f}")


    #print(analysis['bounds'])
                               
    all_pars = sample.getParamNames().list()
                               
    analysis['Sampled points']         = pd.DataFrame(sample.makeSingleSamples(),columns=all_pars)
    analysis['Sampled points']['Case'] = name
                    
    return analysis

In [ ]:
print(obs)

In [ ]:
analyzed_results = {ob: {name: analyze_results(name,resdict) for name,resdict in results[ob].items() } for ob in obs}


In [ ]:
relative_error=(deepcopy(analyzed_results))
for ob in obs:
    for name in analyzed_results[ob].keys():
        for par in plot_pars:
            relative_error[ob][name]['GW_configuration'][par] = analyzed_results[ob][name]['relative_error_percent'][par] 


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Function to create matrix plot for a specific parameter and observation
def plot_relative_error_heatmap(relative_error, obs, param_name, N_gw_configurations, sigma_dL_configurations):
    for ob in obs:  # Iterate over observations
        # Initialize matrix to store relative errors
        error_matrix = np.zeros((len(sigma_dL_configurations), len(N_gw_configurations)))

        # Populate the matrix with relative errors
        for i, sigma_dL in enumerate(sigma_dL_configurations):
            for j, N_gw in enumerate(N_gw_configurations):
                # Generate the key for the case
                case = f"$N_gw$={int(N_gw)}, $\sigma$={sigma_dL}, LSS $\times$ {ob}"
                error_matrix[i, j] = relative_error[ob][case]['GW_configuration'][param_name]

        # Create the heatmap
        fig, ax = plt.subplots(figsize=(8, 6))
        im = ax.imshow(error_matrix, cmap='RdPu', aspect='auto', interpolation='nearest')

        # Set axis labels
        ax.set_xticks(range(len(N_gw_configurations)))
        ax.set_xticklabels([
            f"${int(N_gw / 10**int(np.log10(N_gw)))} \\cdot 10^{{{int(np.log10(N_gw))}}}$" 
            if N_gw / 10**int(np.log10(N_gw)) != 1 else f"$10^{{{int(np.log10(N_gw))}}}$"
            for N_gw in N_gw_configurations
        ])
        ax.set_yticks(range(len(sigma_dL_configurations)))
        ax.set_yticklabels([f"{sigma_dL*100:.1f}%" for sigma_dL in sigma_dL_configurations])

        # Add title and labels
        ax.set_title(f"Relative Error for {param_name} - {ob}", fontsize=14)
        ax.set_xlabel("$N_{GW}$", fontsize=12)
        ax.set_ylabel("$\\sigma_{dL}^{gw} \%$", fontsize=12)

        for i in range(len(sigma_dL_configurations)):
            for j in range(len(N_gw_configurations)):
                value = error_matrix[i, j] * 100  # Convert to percentage
                ax.text(j, i, f"{value:.2f}\%", ha="center", va="center", color="white" if im.norm(error_matrix[i, j]) > 0.5 else "black")

        # Tight layout and show the plot
        plt.tight_layout()
        plt.show()

# Example Usage
plot_relative_error_heatmap(
    relative_error=relative_error,
    obs=['GWC', 'GWWL', 'GWs'],  # List of observations
    param_name='mu0',  # Parameter for which to plot the relative error
    N_gw_configurations=[1e4, 5e4, 1e5, 5e5, 1e6],  # N_gw configurations
    sigma_dL_configurations=[0.1, 0.05, 0.01, 0.005]  # sigma_dL configurations
)


In [ ]:
plot_relative_error_heatmap(
    relative_error=relative_error,
    obs=['GWC', 'GWWL', 'GWs'],  # List of observations
    param_name='sigma0',  # Parameter for which to plot the relative error
    N_gw_configurations=[1e4, 5e4, 1e5, 5e5, 1e6],  # N_gw configurations
    sigma_dL_configurations=[0.1, 0.05, 0.01, 0.005]  # sigma_dL configurations
)

In [ ]:
plot_relative_error_heatmap(
    relative_error=relative_error,
    obs=['GWC', 'GWWL', 'GWs'],  # List of observations
    param_name='omch2',  # Parameter for which to plot the relative error
    N_gw_configurations=[1e4, 5e4, 1e5, 5e5, 1e6],  # N_gw configurations
    sigma_dL_configurations=[0.1, 0.05, 0.01, 0.005]  # sigma_dL configurations
)

In [ ]:
plot_relative_error_heatmap(
    relative_error=relative_error,
    obs=['GWC', 'GWWL', 'GWs'],  # List of observations
    param_name='logA',  # Parameter for which to plot the relative error
    N_gw_configurations=[1e4, 5e4, 1e5, 5e5, 1e6],  # N_gw configurations
    sigma_dL_configurations=[0.1, 0.05, 0.01, 0.005]  # sigma_dL configurations
)